# Pre-Processing of OpenPose Data by Zac Stritch-Hoddle

## Get Facial Key-Points and Save as ".csv" File

In [ ]:
import os
import json
import pandas as pd

in_directory = ""
out_directory = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Processed Pose Data"

# Ensure the output directory exists
if not os.path.exists(out_directory):
    os.makedirs(out_directory)

# Iterate through all JSON files in the input directory
for json_file_name in os.listdir(in_directory):
    if json_file_name.endswith('.json'):
        print(f"Processing {json_file_name}...")
        json_file_path = os.path.join(in_directory, json_file_name)

        # Load JSON data
        with open(json_file_path, 'r') as file:
            data = json.load(file)

        # Initialize a list to hold the rows of the DataFrame
        rows = []

        # Iterate through each frame in the JSON data
        for frame in data:
            if "people" in frame and len(frame["people"]) > 0:
                face_keypoints = frame["people"][0]["face_keypoints_2d"]
                # Ensure that we have exactly 210 values (70 keypoints * 3 values per keypoint)
                if len(face_keypoints) == 210:
                    rows.append(face_keypoints)

        # Convert the list of rows into a DataFrame
        if rows:
            df = pd.DataFrame(rows)

            # Generate column names (e.g., "x1", "y1", "prob1", "x2", "y2", "prob2", ..., "x70", "y70", "prob70")
            columns = []
            for i in range(70):
                columns.extend([f'x{i+1}', f'y{i+1}', f'prob{i+1}'])
            df.columns = columns

            # Save the DataFrame to a CSV file
            out_file_name = json_file_name.replace('combined.json', 'pose.csv')
            out_file_path = os.path.join(out_directory, out_file_name)
            df.to_csv(out_file_path, index=False)

            print(f"CSV file has been saved to {out_file_path}")
        else:
            print(f"No valid face keypoints found in {json_file_name}")


## Extract and Filter Centre-Face (Head), Right and Left Eye, and Right and Left Pupil Position.

In [1]:
import os
import pandas as pd
from scipy.signal import butter, filtfilt

def load_csv(file_path):
    return pd.read_csv(file_path)

def compute_averages(df):
    # Extract specific columns for each set of points
    center_face_x = df[[f'x{i}' for i in range(27, 36)]].mean(axis=1)
    center_face_y = df[[f'y{i}' for i in range(27, 36)]].mean(axis=1)
    center_face_prob = df[[f'prob{i}' for i in range(27, 36)]].mean(axis=1)
    
    left_eye_x = df[[f'x{i}' for i in range(36, 42)]].mean(axis=1)
    left_eye_y = df[[f'y{i}' for i in range(36, 42)]].mean(axis=1)
    left_eye_prob = df[[f'prob{i}' for i in range(36, 42)]].mean(axis=1)
    
    right_eye_x = df[[f'x{i}' for i in range(42, 48)]].mean(axis=1)
    right_eye_y = df[[f'y{i}' for i in range(42, 48)]].mean(axis=1)
    right_eye_prob = df[[f'prob{i}' for i in range(42, 48)]].mean(axis=1)
    
    left_pupil_x = df['x68']
    left_pupil_y = df['y68']
    left_pupil_prob = df['prob68']
    
    right_pupil_x = df['x69']
    right_pupil_y = df['y69']
    right_pupil_prob = df['prob69']

    # Combine the averages into a DataFrame
    averaged_df = pd.DataFrame({
        'center_face_x': center_face_x,
        'center_face_y': center_face_y,
        'center_face_prob': center_face_prob,
        'left_eye_x': left_eye_x,
        'left_eye_y': left_eye_y,
        'left_eye_prob': left_eye_prob,
        'right_eye_x': right_eye_x,
        'right_eye_y': right_eye_y,
        'right_eye_prob': right_eye_prob,
        'left_pupil_x': left_pupil_x,
        'left_pupil_y': left_pupil_y,
        'left_pupil_prob': left_pupil_prob,
        'right_pupil_x': right_pupil_x,
        'right_pupil_y': right_pupil_y,
        'right_pupil_prob': right_pupil_prob,
    })

    return averaged_df

# a functrion that takes in averaged_df and for each *_x and *_y columns, gets the x-y vector magnitude
def compute_magnitude(df):
    for column in df.columns:
        if column.endswith('_x'):
            x_column = column
            y_column = column.replace('_x', '_y')
            magnitude_column = column.replace('_x', '_magnitude')
            df[magnitude_column] = (df[x_column]**2 + df[y_column]**2)**0.5
    return df

def process_data(in_directory, out_directory):
    if not os.path.exists(out_directory):
        os.makedirs(out_directory)
    
    for csv_file_name in os.listdir(in_directory):
        if csv_file_name.endswith('_pose.csv'):
            csv_file_path = os.path.join(in_directory, csv_file_name)
            
            # Load CSV data
            df = load_csv(csv_file_path)
            
            # Compute averages
            averaged_df = compute_averages(df)

            # Compute magnitude
            averaged_df = compute_magnitude(averaged_df)
            
            # If data length is less than 30,000 rows
            # data is at 30 hz, upsample to 60 hz
            if len(averaged_df) < 30000:
                averaged_df = averaged_df.reindex(averaged_df.index.repeat(2)).reset_index(drop=True)
            
            # Save the filtered data to a new CSV file
            out_file_name = csv_file_name.replace('_pose.csv', '_face_eye.csv')
            out_file_path = os.path.join(out_directory, out_file_name)
            averaged_df.to_csv(out_file_path, index=False)
            
            print(f"Extracted data has been been saved to {out_file_name}")

# Directory paths
in_directory = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3_openpose60Hz'
out_directory = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3_EyeFaceData'

# Process and save filtered data
process_data(in_directory, out_directory)


Extracted data has been been saved to 3101_01_face_eye.csv
Extracted data has been been saved to 3219_03_face_eye.csv
Extracted data has been been saved to 3219_02_face_eye.csv
Extracted data has been been saved to 3237_03_face_eye.csv
Extracted data has been been saved to 3237_02_face_eye.csv
Extracted data has been been saved to 3247_03_face_eye.csv
Extracted data has been been saved to 3247_02_face_eye.csv
Extracted data has been been saved to 3218_01_face_eye.csv
Extracted data has been been saved to 3246_01_face_eye.csv
Extracted data has been been saved to 3236_01_face_eye.csv
Extracted data has been been saved to 3210_02_face_eye.csv
Extracted data has been been saved to 3210_03_face_eye.csv
Extracted data has been been saved to 3223_01_face_eye.csv
Extracted data has been been saved to 3222_02_face_eye.csv
Extracted data has been been saved to 3222_03_face_eye.csv
Extracted data has been been saved to 3211_01_face_eye.csv
Extracted data has been been saved to 3208_01_face_eye.c

## Rename Files Such That the Difficulty Condition Corresponding to Each Block is in the File Name.

In [1]:
import os
import pandas as pd

# Load the CSV file
csv_file = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Processed/exp3_session_condition_info.csv'
data = pd.read_csv(csv_file)

# Convert the data into a dictionary for easy lookup
difficulty_map = {}
for _, row in data.iterrows():
    participant_id = str(row['PartID'])
    difficulty_map[participant_id] = {
        f"{i+1:02d}": row[str(i)] for i in range(3)  # Maps block numbers (01, 02, etc.) to difficulties
    }

# Directory containing the files
folder_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData'

# Rename files
for filename in os.listdir(folder_path):
    if filename.endswith("_face_eye.csv"):
        # Extract participant ID and block from the filename
        parts = filename.split('_')
        participant_id = parts[0]
        block = parts[1]

        # Get the corresponding difficulty condition
        difficulty = difficulty_map.get(participant_id, {}).get(block, "Unknown")

        # Create the new filename
        new_filename = f"{participant_id}_{block}_{difficulty}.csv"

        # Rename the file
        old_path = os.path.join(folder_path, filename)
        new_path = os.path.join(folder_path, new_filename)
        os.rename(old_path, new_path)

        print(f"Renamed: {filename} -> {new_filename}")


Renamed: 3249_02_face_eye.csv -> 3249_02_M3.csv
Renamed: 3231_01_face_eye.csv -> 3231_01_L3.csv
Renamed: 3233_02_face_eye.csv -> 3233_02_H2.csv
Renamed: 3222_03_face_eye.csv -> 3222_03_M1.csv
Renamed: 3206_03_face_eye.csv -> 3206_03_H1.csv
Renamed: 3217_02_face_eye.csv -> 3217_02_H1.csv
Renamed: 3215_01_face_eye.csv -> 3215_01_L2.csv
Renamed: 3241_03_face_eye.csv -> 3241_03_H3.csv
Renamed: 3228_01_face_eye.csv -> 3228_01_L2.csv
Renamed: 3250_02_face_eye.csv -> 3250_02_H3.csv
Renamed: 3234_03_face_eye.csv -> 3234_03_M3.csv
Renamed: 3218_02_face_eye.csv -> 3218_02_H1.csv
Renamed: 3209_03_face_eye.csv -> 3209_03_H1.csv
Renamed: 3225_02_face_eye.csv -> 3225_02_M2.csv
Renamed: 3227_01_face_eye.csv -> 3227_01_L2.csv
Renamed: 3246_02_face_eye.csv -> 3246_02_H2.csv
Renamed: 3102_01_face_eye.csv -> 3102_01_L1.csv
Renamed: 3244_01_face_eye.csv -> 3244_01_L1.csv
Renamed: 3210_03_face_eye.csv -> 3210_03_H1.csv
Renamed: 3217_03_face_eye.csv -> 3217_03_M2.csv
Renamed: 3243_01_face_eye.csv -> 3243_01